In [ ]:
pip install rectools

  Using cached rectools-0.18.0-py3-none-any.whl (233 kB)
  Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.2 MB)
  Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)
  Using cached pm_implicit-0.7.3-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (12.8 MB)
  Using cached attrs-23.2.0-py3-none-any.whl (60 kB)
  Using cached fastrlock-0.8.3-cp310-cp310-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_28_x86_64.whl (53 kB)
  Using cached typeguard-4.5.1-py3-none-any.whl (36 kB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
  Attempting uninstall: attrs
    Found existing installation: attrs 26.1.0
    Uninstalling attrs-26.1.0:
      Successfully uninstalled attrs-26.1.0


In [ ]:
import os
import warnings
import json

In [2]:
import torch
import pandas as pd

In [ ]:
from lightning_fabric import seed_everything
from pytorch_lightning import Trainer
from pytorch_lightning.loggers import CSVLogger
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from rectools.dataset import Dataset

In [ ]:
from rectools.models import HSTUModel

In [ ]:
from rectools import Columns
from rectools.model_selection.splitter import Splitter
from rectools.model_selection import LastNSplitter
from rectools.metrics import (
    CoveredUsers,
    Serendipity,
    NDCG,
    AvgRecPopularity,
    CatalogCoverage,
    Recall,
    SufficientReco,
)

In [7]:
from rectools.models import  SASRecModel
from rectools.model_selection import  cross_validate
from rectools.models.nn.item_net import IdEmbeddingsItemNet
from rectools.models.nn.transformers.utils import  leave_one_out_mask

ModuleNotFoundError: No module named 'rectools'

In [ ]:
from utils import  RecallCallback, BestModelLoadCallback, get_results

In [ ]:
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", FutureWarning)

In [ ]:
RANDOM_STATE=42
torch.use_deterministic_algorithms(True)
seed_everything(RANDOM_STATE, workers=True)

In [ ]:
RECALL_K = 10
PATIENCE = 5
DIVERGENCE_TRESHOLD = 0.01
EPOCHS = 10
recall_callback = RecallCallback(k=RECALL_K, progress_bar=True)
# Checkpoints based on best recall
max_recall_ckpt = ModelCheckpoint(
    monitor=f"recall@{RECALL_K}",   # or just pass "val_loss" here,
    mode="max",
    filename="best_recall",
)
early_stopping_recall = EarlyStopping(
    monitor=f"recall@{RECALL_K}",
    mode="max",
    patience=PATIENCE,
    divergence_threshold=DIVERGENCE_TRESHOLD,
)
best_model_load = BestModelLoadCallback("best_recall")
callbacks = [recall_callback, max_recall_ckpt, best_model_load]

# Function to get custom trainer
def get_trainer() -> Trainer:
    return Trainer(
        accelerator="gpu",
        devices=1,
        min_epochs=10,
        max_epochs=EPOCHS,
        deterministic=True,
        enable_model_summary=False,
        enable_progress_bar=True,
        callbacks=callbacks,
        logger = CSVLogger("test_logs"),  # We use CSV logging for this guide but there are many other options
    )

In [ ]:
loo_splitter = LastNSplitter(n=1, n_splits=1, filter_cold_users = False, filter_cold_items = False)

In [ ]:
# Prepare test metrics

metrics_add = {}
metrics_recall ={}
metrics_ndcg = {}
k_base =  10
K = [10, 50,100,200]
K_RECS= max(K)
for k in K:
    metrics_recall.update({
            f"recall@{k}": Recall(k=k),
        })
    metrics_ndcg.update({
            f"ndcg@{k}": NDCG(k=k, divide_by_achievable=True),
        })
metrics_add = {
    f"arp@{k_base}": AvgRecPopularity(k=k_base, normalize=True),
    f"coverage@{k_base}": CatalogCoverage(k=k_base, normalize=True),
    f"covered_users@{k_base}": CoveredUsers(k=k_base),
    f"sufficient_reco@{k_base}": SufficientReco(k=k_base),
    f"serendipity@{k_base}": Serendipity(k=k_base),
}
metrics  = metrics_recall | metrics_ndcg | metrics_add
metrics_to_show = ['recall@10', 'ndcg@10', 'recall@50', 'ndcg@50', 'recall@200', 'ndcg@200', 'coverage@10',
                       'serendipity@10']

In [ ]:
def evaluate(models: dict, splitter:Splitter,dataset: Dataset, path_to_save_res:str) -> None:
    cv_results = cross_validate(
        dataset=dataset,
        splitter=splitter,
        models=models,
        metrics=metrics,
        k=K_RECS,
        filter_viewed=True,
    )
    cv_results["models_log_dir"] = {}
    for model_name, model in models.items():
        cv_results["models_log_dir"].update({model_name:model.fit_trainer.log_dir})
    with open(path_to_save_res, 'w', encoding='utf-8') as f:
        json.dump(cv_results, f, ensure_ascii=False, indent=4)

In [ ]:
config = {
    "session_max_len": 200,
    "lightning_module_kwargs": {"logits_t": 0.05}, # logits scale factor same as in the original repository
    "item_net_block_types": (IdEmbeddingsItemNet,),
    "get_val_mask_func": leave_one_out_mask, # validation mask
    "get_trainer_func": get_trainer,
    "verbose": 1,
    "loss": 'sampled_softmax',
    "n_negatives": 128,
    "use_pos_emb": True,
    "dropout_rate": 0.2,
    "n_factors": 50, # embedding dim
    "n_heads": 1,
    "n_blocks": 2,
    "lr": 0.001,
    "batch_size": 128,
}

In [1]:
import glob
from collections import namedtuple

import pandas as pd
import numpy as np
import scipy.stats as ss

import matplotlib.pyplot as plt

pd.set_option("display.precision", 3)

%matplotlib inline

In [2]:
!pip install scipy matplotlib

In [3]:
data = pd.concat([
    pd.read_json(data_path, lines=True)
    for data_path
    in ["/tmp/runAB_I2I_HSTU/botify-log-1/data.json", "/tmp/runAB_I2I_HSTU/botify-log-2/data.json"]
])

In [4]:
data

,message,timestamp,user,track,time,latency,recommendation,experiments
0,next,2026-04-26 16:05:00.823,3255,5091,1.00,0.146405,3383.0,{'HSTU': 'T1'}
1,next,2026-04-26 16:05:02.065,3255,6895,0.13,0.466582,2708.0,{'HSTU': 'T1'}
2,next,2026-04-26 16:05:02.244,3255,9031,0.19,0.003283,3642.0,{'HSTU': 'T1'}
3,next,2026-04-26 16:05:02.965,7229,15,1.00,0.008777,14.0,{'HSTU': 'C'}
4,next,2026-04-26 16:05:02.994,7229,11,0.63,0.002353,2723.0,{'HSTU': 'C'}
...,...,...,...,...,...,...,...,...
10709,next,2026-04-26 17:41:50.161,1535,8983,1.00,0.001481,8363.0,{'HSTU': 'T1'}
10710,next,2026-04-26 17:41:50.171,1535,13157,0.11,0.001488,1067.0,{'HSTU': 'T1'}
10711,next,2026-04-26 17:41:50.181,1535,11728,0.03,0.001888,7861.0,{'HSTU': 'T1'}
10712,next,2026-04-26 17:41:50.190,1535,3538,0.43,0.001508,3061.0,{'HSTU': 'T1'}


In [5]:
data.to_csv('../hw/hw2/data.csv', index=False)

In [ ]:
data.rename(columns={"user": "user_id", "track": "item_id", "timestamp": "datetime"}, inplace=True)

In [ ]:
data['weight'] = 1.0

In [ ]:
dataset_tracks = Dataset.construct(data)

In [ ]:
dataset_tracks

In [ ]:
hstu_tracks  = HSTUModel(
    relative_time_attention=True,
    relative_pos_attention=True,
    **config
)
models_tracks = {
    "hstu_tracks": hstu_tracks,
}

In [ ]:
dataset_name_tracks = "tracks"
pivot_name_tracks = f"pivot_results_{dataset_name_tracks}.json"

In [ ]:
evaluate(models_tracks, loo_splitter, dataset_tracks, pivot_name_tracks)

In [ ]:
from rectools.dataset.context import get_context

In [ ]:
users = sorted(data['user_id'].unique())

In [ ]:
users_datetime = data.groupby(['user_id'])['datetime'].max().reset_index()

In [ ]:
users_dataframe

In [ ]:
context_all_users = get_context(users_datetime)

In [ ]:
%%time
recs_all_users_hstu_tracks_large = hstu_tracks.recommend(
    users, 
    dataset_tracks, 
    k=100, 
    filter_viewed=True, 
    context=context_all_users
)

In [ ]:
recs_all_users_hstu_tracks_large

In [ ]:
recs_dict = recs_all_users_hstu_tracks_large.groupby(["user_id"]).agg({"item_id": list}).to_dict()['item_id']

In [ ]:
with open("hw_recommendations.json", "w") as rf:
    for user, recs in recs_dict.items():
        recommendation = {
            "user": int(user),
            "tracks": recs
        }
        
        rf.write(json.dumps(recommendation) + "\n")